# Preprocessing: Tafel Slopes & Overpotentials from Raw CV Data

Reads raw cyclic voltammetry (CV) files from the HT-SDC instrument and derives
all targets from the **faradaic current** of the 5th cycle: at 50 mV/s the
recorded current is faradaic + capacitive (±C·ν with the sweep direction), so
forward and reverse sweeps are averaged at matched potential to cancel the
charging term. iR correction uses the per-spot EIS resistance, and potentials are converted
from the Ag/AgCl reference to the RHE scale (0.1 M H₂SO₄, pH ≈ 1:
`E_REF_VS_RHE` ≈ +0.256 V), so η values are true overpotentials.

The Tafel slope is fitted with a **two-regime window policy**. The default is
a fixed common window (1.8–5.2 mA/cm², ≈ 0.46 decades) that every extracted
OER branch covers, so slopes are comparable by construction. A minority of
branches (~30%) contain a near-vertical step inside that window — a local
limiting-current plateau separating two waves — and a line fitted across a
step is not a Tafel slope; for those samples the same-width window is
re-anchored just above the step, onto their quasi-linear second wave. Both
regimes are rule-derived and deterministic; the CSV records the regime, the
actual window bounds, the fit R², and a symmetric window-curvature flag, so
every slope is auditable against its plots.

Outputs:

- `composition_Tafel.csv` — quaternary composition + Tafel slope (V/decade),
  fit R², point count, half-window slope ratio, curvature flag, regime
  (`fixed` / `shifted` / `shifted-short` / `unresolvable`), window bounds
  (mA/cm²), and source file name
- `composition_eta.csv`   — quaternary composition + η at 10 mA/cm² (faradaic,
  interpolated; NaN when a sample never reaches 10 mA/cm²)

Differential Tafel (dJ/dV) CSVs — faradaic current vs applied V — and
per-sample plots are saved to `dJdV/`.


In [1]:
import zipfile
import os

import numpy as np
import pandas as pd
import matplotlib

matplotlib.use('Agg')  # Headless backend: figures are saved, never displayed.
import matplotlib.pyplot as plt

from oer_utils import (
    load_catalyst_summary,
    list_data_files,
    match_composition,
    composition_values,
)

# ── Physical constants ───────────────────────────────────────
Vo = 1.23  # OER equilibrium potential vs RHE (V) at 25 °C.
AREA = 0.165  # Electrode area (cm²)
# Reference conversion to the RHE scale. The DTA files log
# "Vf (V vs. Ref.)" against the cell's Ag/AgCl reference; the
# electrolyte is 0.1 M H₂SO₄ (pH ≈ 1.0 including the partial second
# dissociation). E(RHE) = E(Ag/AgCl) + E°(Ag/AgCl vs SHE) + 0.0592·pH
# at 25 °C, so the true overpotential is
# η = V + E_REF_VS_RHE − I·R − Vo. E° below assumes a saturated-KCl
# filling (+0.197 V); a 3 M or 3.5 M KCl filling would shift all η by
# +13 or +8 mV — adjust E_AGCL_VS_SHE if the filling differs.
E_AGCL_VS_SHE = 0.197  # (V) Ag/AgCl, saturated KCl, 25 °C.
PH_ELECTROLYTE = 1.0  # 0.1 M H₂SO₄.
E_REF_VS_RHE = E_AGCL_VS_SHE + 0.0592 * PH_ELECTROLYTE  # ≈ +0.256 V

# ── OER-branch extraction: vertical line test ────────────────
# Trace the smoothed scan left to right in η vs log₁₀(J). η is
# monotone in scan order, so the curve's slope passes through infinity
# exactly where J reverses direction. For the S-shaped pre-OER oxide
# fold, the first vertical tangent (J rising → falling) is the
# lower-right fold — the wrong vertical line — and the second
# (falling → rising) is the upper-left fold. The OER branch is
# everything after the second. A scan with no second vertical tangent
# is single-valued and is used whole. Branch extraction only removes
# the fold; the Tafel region is then located on the branch by its own
# conditions (see tafel_sliding_window). Noise robustness comes from
# persistence
# only: a direction reversal must last TURN_MIN_RUN points to count.
TURN_MIN_RUN = 9  # Persistence (scan points); the oxide fold spans
#                   ~100 points, instrument wiggles far fewer.
ETA_MIN = 0.30  # Tafel validity floor (V), on the true RHE-referenced
#   overpotential scale. Below ~0.3 V these surfaces are still in the
#   oxide-formation / mixed-control region: the oxide–OER valley sits
#   at η ≈ 0.20–0.26 across the dataset and OER current emerges just
#   above it, while the back reaction is negligible far below this
#   (η ≫ RT/F). Numerically this floor matches the validated fits
#   from before the reference correction (0.05 V on the old scale
#   ≈ 0.31 V true).
SNR_MIN = 2.0  # Background-corrected J must exceed this multiple of the
#                background to count as OER signal.

# ── Fixed-window Tafel fit on the extracted branch ───────────
# Every sample is fitted over the same current-density window, so the
# reported slope is an operational Tafel slope over a stated window —
# comparable by construction across the library, which is what an ML
# target needs. The window is the widest round-number interval covered
# by every extracted OER branch in this dataset (branch starts reach up
# to 1.78 mA/cm² and branch ends reach down to 5.30 mA/cm²), giving
# 100% branch coverage. Samples whose window is bent by transport are
# flagged, never dropped; the target is still never fabricated (a fit
# needs real branch points inside the window).
J_WINDOW_LO = 1.8e-3  # A/cm² — lower edge of the common fit window.
J_WINDOW_HI = 5.2e-3  # A/cm² — upper edge (window ≈ 0.46 decades).
X_WINDOW_LO = np.log10(J_WINDOW_LO)
X_WINDOW_HI = np.log10(J_WINDOW_HI)
WINDOW_MIN_PTS = 10  # Fewest branch points inside the window for a fit.
CURVE_TOL = 0.30  # Symmetric window-curvature flag: split the fitted
#                   window at its midpoint and fit a slope to each
#                   half; a half-slope ratio outside
#                   [1/(1+tol), 1+tol] marks the window as curved
#                   (concave-up = transport-like, concave-down =
#                   exiting a transition).
HALF_MIN_PTS = 5  # Fewest points per half-window for the curvature test.

# ── Step detection: two-regime window placement ──────────────
# A minority of branches contain a near-vertical step inside the
# common window: a local limiting-current plateau (current stalls
# while potential climbs) separating two waves. A line fitted across
# a step is not a Tafel slope. When a step is detected inside the
# common window, the same-width window is re-anchored just above the
# step, where the sample's quasi-linear second wave lives; when too
# little branch remains above the step, the fixed-window fit is kept
# and the step is marked unresolvable.
STEP_THRESH = 0.6  # V/decade — a local slope above this marks a step;
#                    no electrode kinetics produce 600 mV/dec, so
#                    steeper segments are steps or artifacts.
STEP_MARGIN = 0.02  # Decades above the step end before the new window.
MIN_SHIFT_DECADES = 0.25  # Shortest acceptable post-step window; the
#                           full width is used when the branch allows.
PROFILE_PTS = 7  # Points per moving fit for the local-slope profile.

# ── Paths ────────────────────────────────────────────────────
ZIP_PATH = 'data/CV.zip'
CV_DIR = 'data/CV/'
DJDV_DIR = 'data/dJdV/'
IMAGE_DIR = 'images/'
TAFEL_DIR = 'images/Tafel/'  # Journal-style plots: fit region only.
TAFEL_DIAG_DIR = 'images/TafelDiagnostics/'  # Full curve + detected region.

# Unzip the CV archive once; it extracts a CV/ folder into data/.
if not os.path.exists(CV_DIR):
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall('data')


In [2]:
def faradaic_curve(V, I, R):
    """Estimate the faradaic η–J curve from a full CV cycle.

    The raw scan is measured at 50 mV/s, so the recorded current is
    the faradaic current plus a capacitive/pseudocapacitive term
    ±C·ν whose sign follows the sweep direction; on the forward sweep
    it inflates the current (and creates the pre-OER fold), on the
    reverse sweep it deflates it. Averaging the two sweeps at matched
    iR-corrected potential cancels the symmetric charging term to
    first order, leaving the faradaic current a Tafel analysis needs.
    Each sweep is iR-corrected with its own instantaneous current
    before matching.

    Parameters
    ----------
    V, I : full CURVE5 arrays (forward then reverse), volts vs the
        reference scale and amperes.
    R : series resistance (Ω) from EIS.

    Returns
    -------
    (eta, J) on the forward-sweep grid, or ``None`` when the cycle
    has no usable reverse sweep.
    """
    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)
    apex = int(np.argmax(V))
    if apex < 100 or len(V) - apex < 100:
        return None
    eta_f = V[: apex + 1] - I[: apex + 1] * R + E_REF_VS_RHE - Vo
    J_f = I[: apex + 1] / AREA
    eta_r = V[apex + 1 :] - I[apex + 1 :] * R + E_REF_VS_RHE - Vo
    J_r = I[apex + 1 :] / AREA
    order = np.argsort(eta_r)
    J_r_matched = np.interp(eta_f, eta_r[order], J_r[order])
    return eta_f, 0.5 * (J_f + J_r_matched)


def faradaic_vs_applied(V, I):
    """Faradaic current vs applied potential, for the dJ/dV curves.

    Same charging-current cancellation as ``faradaic_curve`` but with
    the sweeps matched at the applied potential rather than the
    iR-corrected one, because the dJ/dV files are keyed on the raw V
    axis. At matched applied V the forward and reverse points differ
    slightly in true (iR-corrected) potential, so the cancellation is
    first-order; the residual is second order in R·dI.

    Returns (V_forward, J_faradaic) in (V, A/cm²), or ``None`` when
    the cycle has no usable reverse sweep.
    """
    V = np.asarray(V, dtype=float)
    I = np.asarray(I, dtype=float)
    apex = int(np.argmax(V))
    if apex < 100 or len(V) - apex < 100:
        return None
    V_f, I_f = V[: apex + 1], I[: apex + 1]
    V_r, I_r = V[apex + 1 :], I[apex + 1 :]
    order = np.argsort(V_r)
    I_r_matched = np.interp(V_f, V_r[order], I_r[order])
    return V_f, 0.5 * (I_f + I_r_matched) / AREA


def eta_at_current_density(eta, J, j_target=0.010):
    """η at a target faradaic current density, by interpolation.

    The crossing is taken on the terminal rise: the last contiguous
    run of points at or above ``j_target`` (A/cm²), so residual fold
    structure lower on the curve cannot supply a false crossing.
    """
    eta = np.asarray(eta, dtype=float)
    J = np.asarray(J, dtype=float)
    Js = pd.Series(J).rolling(5, center=True, min_periods=1).median().values
    above = Js >= j_target
    if not above.any():
        return np.nan
    idx = np.where(above)[0]
    splits = np.where(np.diff(idx) > 1)[0] + 1
    first = np.split(idx, splits)[-1][0]  # start of the terminal run
    if first == 0:
        return float(eta[0])
    j0, j1 = Js[first - 1], Js[first]
    if j1 == j0:
        return float(eta[first])
    frac = (j_target - j0) / (j1 - j0)
    return float(eta[first - 1] + frac * (eta[first] - eta[first - 1]))


def find_second_turning_point(Js):
    """Scan index of the second vertical tangent (the upper-left fold).

    Vertical line test, applied to the curve as plotted: in η–log₁₀(J)
    coordinates η is monotone in scan order, so the curve's slope
    passes through infinity exactly where J reverses direction, and
    the plot only contains points with J > 0. Tracing the smoothed
    scan left to right, the first vertical tangent (J rising →
    falling) is the lower-right fold of the S — the wrong vertical
    line — and the second (falling → rising) is the upper-left fold,
    where the OER branch begins.

    Robustness comes from persistence alone, with no magnitude
    thresholds of any kind:
    a direction run must last at least ``TURN_MIN_RUN`` points to
    count, shorter wiggles are absorbed into the run before them, and
    detection is restricted to the trailing stretch of positive
    current (points that appear on the log plot). The second turning
    point is located as the start of the rising run that carries the
    curve to its terminal maximum — on an S-shaped scan this is
    identical to counting two reversals from the left, but it cannot
    be faked by noise in the low-current region. Returns ``None`` when
    the curve never folds (single-valued): no rising run contains the
    maximum, or that run extends back to the start of the trace.
    """
    Js = np.asarray(Js, dtype=float)

    # The plotted curve: trailing contiguous stretch of J > 0.
    nonpos = np.where(Js <= 0)[0]
    i0 = nonpos[-1] + 1 if len(nonpos) else 0
    seg = Js[i0:]
    if len(seg) < 3 * TURN_MIN_RUN:
        return None

    d = np.diff(seg)
    sign = np.sign(d)
    nz = np.nonzero(sign)[0]
    if len(nz) == 0:
        return None
    # Flat stretches inherit the direction of the preceding motion.
    sign[: nz[0]] = sign[nz[0]]
    for k in range(1, len(sign)):
        if sign[k] == 0:
            sign[k] = sign[k - 1]

    # Maximal direction runs over diff indices [a, b): points seg[a..b].
    runs = []
    a = 0
    for k in range(1, len(sign) + 1):
        if k == len(sign) or sign[k] != sign[a]:
            runs.append([a, k, int(sign[a])])
            a = k

    # Persistence merge: runs shorter than TURN_MIN_RUN and runs that
    # continue the previous direction are absorbed into the run before
    # them. Kept runs therefore alternate direction.
    kept = []
    for a, b, direction in runs:
        if (b - a) < TURN_MIN_RUN or (kept and kept[-1][2] == direction):
            if kept:
                kept[-1][1] = b
            continue
        kept.append([a, b, direction])
    if not kept:
        return None

    # The terminal rise: the kept rising run containing the global
    # maximum. Its start is the last valley — the second vertical
    # tangent of the S. If that run is the first kept run, the trace
    # never fell on its way up: no fold.
    gmax = int(np.argmax(seg))
    run_idx = None
    for i, (a, b, direction) in enumerate(kept):
        if direction == 1 and a <= gmax <= b:
            run_idx = i
    if run_idx is None or run_idx == 0:
        return None
    return i0 + kept[run_idx][0]


def extract_oer_branch(eta, J):
    """Isolate the single-valued OER branch via the vertical line test.

    A CV in η vs log₁₀(J) coordinates is parametric: the pre-OER oxide
    wave makes J rise, fall, then rise again, tracing an S that fails
    the vertical line test. ``find_second_turning_point`` locates the
    second vertical tangent (the upper-left fold), and the OER branch
    is everything after it. When no second turning point exists the
    curve is single-valued and is used whole. Extraction only removes
    the fold; fitting the Tafel slope on the branch is
    ``tafel_step_aware``'s job.

    The non-OER background (valley of smoothed |J| below ``ETA_MIN``,
    the local minimum between the oxide wave and onset) is subtracted,
    points must satisfy η > ``ETA_MIN`` and corrected J > ``SNR_MIN`` ×
    background; the branch runs from the first sustained run of valid
    points to the peak of the corrected current, and a running-maximum
    envelope guarantees it is single-valued.

    Returns
    -------
    dict with ``x`` (log₁₀ J_OER), ``y`` (η), ``baseline``,
    ``multivalued``, ``turn_index`` (scan index of the second turning
    point, or None), and ``turn_xy`` (its η–log₁₀ J coordinates for
    diagnostics, or None); or ``None`` when no valid branch exists.
    """
    eta = np.asarray(eta, dtype=float)
    J = np.asarray(J, dtype=float)
    if len(J) < 20:
        return None

    Js = pd.Series(J).rolling(7, center=True, min_periods=1).median().values

    # ── Second turning point → branch start ──────────────────
    turn2 = find_second_turning_point(Js)
    multivalued = turn2 is not None
    start0 = turn2 + 1 if multivalued else 0

    turn_xy = None
    if multivalued and Js[turn2] > 0:
        turn_xy = (float(np.log10(Js[turn2])), float(eta[turn2]))

    # ── Background: valley between oxide wave and OER onset ──
    abs_smooth = (
        pd.Series(np.abs(J)).rolling(7, center=True, min_periods=1).median().values
    )
    pre = abs_smooth[eta < ETA_MIN]
    if len(pre) < 5:
        return None
    baseline = pre.min()

    # ── Subtraction + validity gates on the branch ───────────
    # The gates are deliberately gentle: the branch starts at the
    # first sustained run of valid points (not after the last failure
    # anywhere, which lets one noisy point discard the whole branch),
    # ends at the peak of the smoothed corrected current (a faradaic
    # curve can dip slightly near the scan apex from bubble
    # hysteresis), and single-valuedness comes
    # from a running-maximum envelope that drops the few points
    # inside small dips instead of everything before them.
    eta_b, J_b = eta[start0:], J[start0:]
    J_corr = J_b - baseline
    Jc_smooth = (
        pd.Series(J_corr).rolling(7, center=True, min_periods=1).median().values
    )
    ok = (eta_b > ETA_MIN) & (Jc_smooth > SNR_MIN * baseline) & (J_corr > 0)
    start = None
    run = 0
    for i, valid in enumerate(ok):
        run = run + 1 if valid else 0
        if run >= 10:
            start = i - 9
            break
    if start is None:
        return None
    end = start + int(np.argmax(Jc_smooth[start:])) + 1
    idx = np.arange(start, end)
    idx = idx[ok[start:end]]
    if len(idx) < 10:
        return None

    # ── Single-valued branch: running-maximum envelope ───────
    keep = []
    running_max = -np.inf
    for i in idx:
        if Jc_smooth[i] > running_max:
            keep.append(i)
            running_max = Jc_smooth[i]
    keep = np.array(keep)
    if len(keep) < 10:
        return None

    return {
        'x': np.log10(J_corr[keep]),
        'y': eta_b[keep],
        'baseline': baseline,
        'multivalued': multivalued,
        'turn_index': turn2,
        'turn_xy': turn_xy,
    }


def local_slope_profile(x, y, pts=PROFILE_PTS):
    """Moving-window local slope dη/dlog₁₀J along a sorted branch.

    A linear fit over ``pts`` consecutive points, centered on each
    interior point; endpoints where the window does not fit are NaN.
    Used to detect near-vertical steps (local limiting-current
    plateaus), which appear as local slopes far above anything
    electrode kinetics can produce.
    """
    slopes = np.full(len(x), np.nan)
    half = pts // 2
    for i in range(half, len(x) - half):
        xs = x[i - half:i + half + 1]
        if xs[-1] - xs[0] > 1e-6:
            slopes[i] = np.polyfit(xs, y[i - half:i + half + 1], 1)[0]
    return slopes


def _fit_segment(xf, yf):
    """Linear fit plus R² and a symmetric half-split curvature test."""
    slope, intercept = np.polyfit(xf, yf, 1)
    resid = yf - (slope * xf + intercept)
    ss_tot = float(((yf - yf.mean()) ** 2).sum())
    r2 = 1.0 - float((resid**2).sum()) / ss_tot if ss_tot > 0 else np.nan
    x_mid = 0.5 * (xf[0] + xf[-1])
    lo_half = xf <= x_mid
    hi_half = xf >= x_mid
    slope_ratio = np.nan
    curved = False
    if lo_half.sum() >= HALF_MIN_PTS and hi_half.sum() >= HALF_MIN_PTS:
        b_lo = np.polyfit(xf[lo_half], yf[lo_half], 1)[0]
        b_hi = np.polyfit(xf[hi_half], yf[hi_half], 1)[0]
        if b_lo > 0 and b_hi > 0:
            slope_ratio = b_hi / b_lo
            curved = (
                slope_ratio > 1.0 + CURVE_TOL
                or slope_ratio < 1.0 / (1.0 + CURVE_TOL)
            )
        else:
            curved = True  # A non-positive half-slope is curvature too.
    return slope, intercept, r2, slope_ratio, curved


def tafel_step_aware(
    log_j,
    eta,
    x_lo=X_WINDOW_LO,
    x_hi=X_WINDOW_HI,
    min_pts=WINDOW_MIN_PTS,
    step_thresh=STEP_THRESH,
    step_margin=STEP_MARGIN,
    min_shift_decades=MIN_SHIFT_DECADES,
):
    """Fit the Tafel slope with the two-regime window policy.

    Regime ``fixed`` (default): fit over the common window
    [``J_WINDOW_LO``, ``J_WINDOW_HI``], so the slope is an operational
    b over a stated range, comparable across the library.

    Regime ``shifted``: when the local-slope profile exceeds
    ``step_thresh`` anywhere inside the common window, the branch has a
    near-vertical step there and a straight line across it is
    meaningless; the same-width window is re-anchored ``step_margin``
    decades above the top of the step's contiguous run. When the branch
    ends before a full-width window fits, the window is shortened down
    to ``min_shift_decades`` (regime ``shifted-short``); when even that
    much branch is unavailable, the fixed-window fit is kept and the
    regime is ``unresolvable``, which is a do-not-trust marker.

    Both regimes are deterministic functions of the branch, so every
    reported slope comes from a rule-derived, stated current window.
    Quality is reported, never enforced: R² measures linearity inside
    the fitted window, and the symmetric curvature flag marks windows
    whose half-slope ratio departs from 1 in either direction.

    Returns
    -------
    dict with ``slope`` (V/decade), ``intercept``, ``r2``,
    ``n_points``, ``slope_ratio``, ``curved`` (bool), ``regime``,
    ``j_lo_ma`` / ``j_hi_ma`` (fitted window in mA/cm²), plus sorted
    ``x``, ``y`` arrays and a boolean ``fit_mask``; or ``None`` when
    fewer than ``min_pts`` branch points fall inside the common
    window.
    """
    order = np.argsort(log_j)
    x = np.asarray(log_j, dtype=float)[order]
    y = np.asarray(eta, dtype=float)[order]

    common_mask = (x >= x_lo) & (x <= x_hi)
    if common_mask.sum() < min_pts:
        return None

    profile = local_slope_profile(x, y)
    step_pts = np.isfinite(profile) & (profile > step_thresh)
    step_in_window = step_pts & common_mask

    regime = 'fixed'
    fit_mask = common_mask
    if step_in_window.any():
        # Anchor above the contiguous step run that overlaps the
        # window (the run may extend past the window's upper edge).
        step_idx = np.where(step_pts)[0]
        first_in = np.where(step_in_window)[0].min()
        top = step_idx[step_idx >= first_in].max()
        x_anchor = x[top] + step_margin
        x_end = min(x_anchor + (x_hi - x_lo), x[-1])
        shifted_mask = (x >= x_anchor) & (x <= x_end)
        if (
            x_end - x_anchor >= min_shift_decades
            and shifted_mask.sum() >= min_pts
        ):
            fit_mask = shifted_mask
            full_width = x_end - x_anchor >= (x_hi - x_lo) - 1e-9
            regime = 'shifted' if full_width else 'shifted-short'
        else:
            regime = 'unresolvable'

    xf, yf = x[fit_mask], y[fit_mask]
    slope, intercept, r2, slope_ratio, curved = _fit_segment(xf, yf)

    return {
        'slope': slope,
        'intercept': intercept,
        'r2': r2,
        'n_points': int(fit_mask.sum()),
        'slope_ratio': slope_ratio,
        'curved': curved,
        'regime': regime,
        'j_lo_ma': 10 ** xf[0] * 1e3,
        'j_hi_ma': 10 ** xf[-1] * 1e3,
        'x': x,
        'y': y,
        'fit_mask': fit_mask,
    }


In [3]:
# ── Load composition + iR-drop lookup table ──────────────────
catalyst_df = load_catalyst_summary('data/PtPdAuIr_summary.csv')

os.makedirs(DJDV_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)
os.makedirs(TAFEL_DIR, exist_ok=True)
os.makedirs(TAFEL_DIAG_DIR, exist_ok=True)

cv_files = list_data_files(CV_DIR)

regime_counts = {}  # Tally of window regimes, printed after the loop.

with open('data/composition_Tafel.csv', 'w') as tafel_file, open(
    'data/composition_eta.csv', 'w'
) as eta_file:

    tafel_file.write(
        'Pt,Pd,Au,Ir,Tafel slope,Tafel R2,Tafel n,Slope ratio,'
        'Curved window,Regime,Window lo (mA/cm2),'
        'Window hi (mA/cm2),J baseline,File\n'
    )
    eta_file.write('Pt,Pd,Au,Ir,Eta\n')

    for filename in cv_files:
        if filename == '.DS_Store':
            continue

        # ── Match filename → composition row ─────────────────
        cat_row = match_composition(catalyst_df, filename)
        Pt, Pd, Au, Ir = composition_values(cat_row)

        # ── Extract 5th forward scan (LSV) ───────────────────
        cv_df = pd.read_csv(CV_DIR + filename)
        scan_start = 0
        with open(CV_DIR + filename, 'r') as f:
            for line_num, line in enumerate(f, start=1):
                if line[:6] == 'CURVE5':
                    scan_start = line_num + 1  # +1 for header row

        # ── Full 5th cycle: forward + reverse sweep ──────────
        # Every quantity below is computed from the faradaic current.
        # The recorded current at 50 mV/s is faradaic + capacitive
        # (±C·ν, sign set by sweep direction), and the charging term
        # dominates below ~1 mA/cm²; the forward sweep alone is never
        # used as a current source. Files without a usable reverse
        # sweep yield NaN targets rather than contaminated ones.
        cyc = cv_df.iloc[scan_start:, 3:5].copy()
        cyc.columns = ['V', 'I']
        V_cyc = pd.to_numeric(cyc['V'], errors='coerce').values
        I_cyc = pd.to_numeric(cyc['I'], errors='coerce').values
        good = np.isfinite(V_cyc) & np.isfinite(I_cyc)
        V_cyc, I_cyc = V_cyc[good], I_cyc[good]

        R = cat_row['iR drop (Ω)'].values[0]
        fc_eta = faradaic_curve(V_cyc, I_cyc, R)
        fc_app = faradaic_vs_applied(V_cyc, I_cyc)

        # ── η at 10 mA/cm² (faradaic, interpolated) ──────────
        if fc_eta is not None:
            eta10 = eta_at_current_density(*fc_eta, j_target=0.010)
        else:
            eta10 = np.nan
        eta_file.write(f'{Pt},{Pd},{Au},{Ir},{eta10}\n')

        # ── Differential Tafel (dJ/dV), faradaic current ─────
        # Here we only
        # compute and store the numerical curves.
        if fc_app is not None:
            dtp = pd.DataFrame({'V': fc_app[0], 'J': fc_app[1]})
            dtp = dtp.groupby('V', as_index=False).mean()
            low_mask = dtp['V'] < 1.0
            dtp.loc[low_mask, 'J'] = (
                dtp.loc[low_mask, 'J'].rolling(window=5, center=True).mean()
            )
            V_arr = dtp['V'].values
            dJdV = np.gradient(dtp['J'].values, V_arr)

            out_path = os.path.join(DJDV_DIR, filename)
            np.savetxt(
                out_path,
                np.column_stack((V_arr, dJdV)),
                delimiter=',',
                header='V,dJdV',
                comments='',
            )

        # ── Tafel slope: faradaic curve + fixed common window ────
        # η(log J) generation: the recorded current at 50 mV/s is
        # faradaic + capacitive (±C·ν, sign set by sweep direction),
        # and the capacitive term dominates the low-current decade of
        # the forward sweep — it is what draws most of the pre-OER
        # fold. The Tafel curve is therefore built from the full 5th
        # cycle by averaging forward and reverse currents at matched
        # iR-corrected potential (faradaic_curve), which cancels the
        # charging term. The branch extractor applies the vertical
        # line test — everything after the second vertical tangent —
        # purely to remove the multivalued fold; when no second
        # vertical tangent exists (on faradaic curves the fold usually
        # cancels away), the curve is single-valued and used whole.
        # The slope is then fitted with the two-regime policy:
        # the fixed common window J_WINDOW_LO–J_WINDOW_HI by default,
        # re-anchored above a detected near-vertical step for the
        # minority of branches that carry one inside the window. R²,
        # the symmetric curvature flag, the regime, and the fitted
        # window bounds record per-sample quality and provenance.
        if fc_eta is not None:
            eta_far, J_far = fc_eta
            branch = extract_oer_branch(eta_far, J_far)
        else:
            eta_far = J_far = None
            branch = None
        fit = (
            tafel_step_aware(branch['x'], branch['y'])
            if branch is not None
            else None
        )
        regime = fit['regime'] if fit is not None else 'no-fit'
        regime_counts[regime] = regime_counts.get(regime, 0) + 1

        if fit is not None:
            tafel_slope = fit['slope']
            tafel_file.write(
                f'{Pt},{Pd},{Au},{Ir},{tafel_slope},'
                f"{fit['r2']},{fit['n_points']},{fit['slope_ratio']},"
                f"{int(fit['curved'])},{fit['regime']},"
                f"{fit['j_lo_ma']},{fit['j_hi_ma']},"
                f"{branch['baseline']},{filename}\n"
            )
        else:
            tafel_slope = np.nan
            bl = branch['baseline'] if branch is not None else np.nan
            tafel_file.write(
                f'{Pt},{Pd},{Au},{Ir},nan,nan,0,nan,0,no-fit,'
                f'nan,nan,{bl},{filename}\n'
            )

        # ── Save plots to disk (never displayed) ─────────────
        # Journal-style plot (images/Tafel/): the Tafel region only,
        # as it would appear in a figure — fit-region data + fit line.
        # Diagnostic plot (images/TafelDiagnostics/): the full curve
        # with the detected region highlighted, for auditing the
        # extraction.
        sample_id = cat_row.iloc[0, 0]
        plot_name = os.path.splitext(filename)[0]
        composition_label = f'Pt({Pt}), Pd({Pd}), Au({Au}), Ir({Ir})'

        if fit is not None:
            xm = fit['x'][fit['fit_mask']]
            ym = fit['y'][fit['fit_mask']]
            xl = np.array([xm[0], xm[-1]])
            fit_label = (
                f"{fit['slope'] * 1000:.0f} mV dec$^{{-1}}$ "
                f"({fit['j_lo_ma']:.1f}–{fit['j_hi_ma']:.1f} "
                f"mA cm$^{{-2}}$)"
            )

            # Journal-style plot: fit region only.
            fig, ax = plt.subplots(figsize=(6, 4.5))
            ax.scatter(xm, ym, s=18, color='black', zorder=2)
            ax.plot(
                xl,
                fit['slope'] * xl + fit['intercept'],
                color='crimson',
                linewidth=1.8,
                zorder=3,
                label=fit_label,
            )
            ax.set_xlabel(r'log$_{10}$ $J_{\mathrm{OER}}$  (A cm$^{-2}$)')
            ax.set_ylabel(r'$\eta$ (V)')
            ax.set_title(composition_label, fontsize=11)
            ax.legend(loc='lower right', frameon=False)
            plt.tight_layout()
            plt.savefig(
                os.path.join(TAFEL_DIR, f'Tafel_{plot_name}.png'), dpi=300
            )
            plt.close('all')

        # Diagnostic plot: full curve + detected region + gates outcome.
        fig, ax = plt.subplots(figsize=(8, 5))
        if J_far is not None:
            pos_far = J_far > 0
            ax.scatter(
                np.log10(J_far[pos_far]),
                eta_far[pos_far],
                s=12,
                color='0.6',
                label='Faradaic curve (fwd/rev averaged)',
            )
        if branch is not None:
            ax.scatter(
                branch['x'],
                branch['y'],
                s=12,
                color='goldenrod',
                label='OER branch (J − background)',
            )
            if branch['turn_xy'] is not None:
                ax.axvline(
                    branch['turn_xy'][0],
                    color='darkorchid',
                    linestyle='-',
                    linewidth=1,
                    alpha=0.6,
                )
                ax.scatter(
                    [branch['turn_xy'][0]],
                    [branch['turn_xy'][1]],
                    marker='*',
                    s=180,
                    color='darkorchid',
                    zorder=5,
                    label='2nd turning point (vertical tangent)',
                )
        ax.axvspan(
            X_WINDOW_LO,
            X_WINDOW_HI,
            color='steelblue',
            alpha=0.08,
            label=(
                f'Common fit window '
                f'({J_WINDOW_LO * 1e3:.1f}–{J_WINDOW_HI * 1e3:.1f} mA/cm²)'
            ),
        )
        if fit is not None and fit['regime'].startswith('shifted'):
            ax.axvspan(
                np.log10(fit['j_lo_ma'] * 1e-3),
                np.log10(fit['j_hi_ma'] * 1e-3),
                color='darkorange',
                alpha=0.12,
                label=(
                    f"Shifted window above step "
                    f"({fit['j_lo_ma']:.1f}–{fit['j_hi_ma']:.1f} mA/cm²)"
                ),
            )
        ax.axhline(ETA_MIN, color='gray', linestyle=':', linewidth=1)
        if fit is not None:
            ax.scatter(xm, ym, s=14, color='steelblue', label='Fit region')
            ax.plot(
                xl,
                fit['slope'] * xl + fit['intercept'],
                color='crimson',
                linewidth=2,
                label=(
                    f"{fit['slope'] * 1000:.0f} mV/dec, "
                    f"R²={fit['r2']:.3f}"
                    + (
                        f", {fit['regime']}"
                        if fit['regime'] != 'fixed'
                        else ''
                    )
                    + (', curved' if fit['curved'] else '')
                ),
            )
        else:
            ax.annotate(
                'Too few points in the common window\n(slope = NaN)',
                xy=(0.05, 0.9),
                xycoords='axes fraction',
                color='crimson',
            )
        ax.set_xlabel(r'log$_{10}$ J  (A/cm²)')
        ax.set_ylabel('η [V]')
        ax.set_title(f'Tafel Diagnostics — {composition_label}')
        ax.legend(loc='best', fontsize=9)
        ax.grid(True, alpha=0.3)
        plt.tight_layout()
        plt.savefig(
            os.path.join(TAFEL_DIAG_DIR, f'TafelDiag_{plot_name}.png'),
            dpi=300,
        )
        plt.close('all')  # Free figure memory each iteration.

        slope_txt = (
            f'{tafel_slope * 1000:.1f} mV/dec'
            + (f" [{fit['regime']}]" if fit['regime'] != 'fixed' else '')
            + (' [curved]' if fit['curved'] else '')
            if fit is not None
            else 'no fit in common window (NaN)'
        )
        eta_txt = f'{eta10:.4f} V' if np.isfinite(eta10) else 'NaN (< 10 mA/cm²)'
        print(f'sample {sample_id}: η₁₀ = {eta_txt}, Tafel slope = {slope_txt}')

# ── Window-regime summary ────────────────────────────────────
total = sum(regime_counts.values())
print('\nWindow regimes:')
for reg in ('fixed', 'shifted', 'shifted-short', 'unresolvable',
            'no-fit'):
    if reg in regime_counts:
        print(f'  {reg:14s} {regime_counts[reg]:4d} '
              f'({100 * regime_counts[reg] / total:.0f}%)')


sample 46: η₁₀ = 0.7844 V, Tafel slope = 351.1 mV/dec [shifted]
sample 55: η₁₀ = 0.7183 V, Tafel slope = 371.1 mV/dec [curved]
sample 56: η₁₀ = 0.5663 V, Tafel slope = 208.7 mV/dec [curved]
sample 57: η₁₀ = 0.5161 V, Tafel slope = 177.3 mV/dec [curved]
sample 58: η₁₀ = NaN (< 10 mA/cm²), Tafel slope = 205.6 mV/dec
sample 59: η₁₀ = NaN (< 10 mA/cm²), Tafel slope = 246.3 mV/dec
sample 60: η₁₀ = 0.8011 V, Tafel slope = 249.0 mV/dec
sample 61: η₁₀ = 0.7313 V, Tafel slope = 329.6 mV/dec [shifted] [curved]
sample 62: η₁₀ = 0.7246 V, Tafel slope = 476.5 mV/dec [unresolvable] [curved]
sample 63: η₁₀ = 0.6487 V, Tafel slope = 283.0 mV/dec [unresolvable] [curved]
sample 64: η₁₀ = 0.4902 V, Tafel slope = 151.8 mV/dec [curved]
sample 47: η₁₀ = 0.7557 V, Tafel slope = 324.8 mV/dec [shifted] [curved]
sample 65: η₁₀ = NaN (< 10 mA/cm²), Tafel slope = 97.3 mV/dec [shifted-short] [curved]
sample 66: η₁₀ = NaN (< 10 mA/cm²), Tafel slope = 221.6 mV/dec
sample 67: η₁₀ = NaN (< 10 mA/cm²), Tafel slope = 21